> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [環境準備](#環境-준비)
- [Resource Group 作成](#resource-group-作成)
- [Foundry リソース 作成](#foundry-リソース-作成)
- [リソース 確認](#リソース-確認)
- [次のステップ](#次へ-단계)

## 🎯 学習目標

- Azure CLI로 Resource Group 作成
- Azure CLI 또는 Bicep으로 Microsoft Foundry リソース 作成
- コードベースのInfrastructure as Code (IaC)実習

## ⏱️ 予想所要時間

約10分

## 環境準備

### 必須要件

1. **Azure CLI 설치 確認**
   - 아래 셀을 実行하여 Azure CLI가 설치되어 있는지 確認합니다.
   - インストールされていない場合: [Azure CLIインストールガイド](https://learn.microsoft.com/cli/azure/install-azure-cli)

2. **Azureログイン**
   - Azure CLIでAzureアカウントにログインします。

## Python 가상環境設定

仮想環境を使用するとプロジェクトごとに独立したPythonパッケージを管理できます。

### 仮想環境とは？

가상環境(Virtual Environment)은 プロジェクト마다 독립적인 Python 実行 環境을 만들어줍니다.

**メリット:**
- プロジェクトごとに異なるバージョンのパッケージを使用可能
- システムPython環境をクリーンに維持
- チームメンバー間で同じ開発環境を共有しやすい

### 가상環境 作成 및 有効化

**1. 터미널에서 가상環境 作成**

macOS / Linux:
```bash
python3 -m venv .venv
```

Windows:
```bash
python -m venv .venv
```

**2. 仮想環境の有効化**

macOS / Linux:
```bash
source .venv/bin/activate
```

Windows (PowerShell):
```powershell
.venv\Scripts\Activate.ps1
```

Windows (CMD):
```cmd
.venv\Scripts\activate.bat
```

**3. 有効化 確認**

有効化されるとターミナルプロンプトの前に`(.venv)`が表示されます:
```
(.venv) user@machine:~/project$
```

### VS Code Jupyter使用時

VS CodeでJupyter Notebookを使用する場合:
1. 가상環境 作成 후
2. 우측 상단 **커널 選択** 버튼 클릭
3. **'.venv' 環境** 選択

이후 모든 셀은 자동으로 가상環境에서 実行됩니다.

In [ ]:
# 必須パッケージのインストール
# 가상環境이 有効化된 状態에서 実行하세요

!pip install -q azure-ai-projects azure-identity azure-mgmt-resource

print("✅ 必須パッケージのインストール 完了!")
print("\n📦 インストールされたパッケージ:")
print("   - azure-ai-projects")
print("   - azure-identity")
print("   - azure-mgmt-resource")
print("\n💡 Azure CLIはシステムレベルで別途インストールが必要です:")
print("   https://learn.microsoft.com/cli/azure/install-azure-cli")

In [ ]:
# PATH 環境変数 設定
# JupyterカーネルでAzure CLIを見つけられるようにパスを追加
import os
import subprocess

# Azure CLI가 설치될 수 있는 여러 パス 確認
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

# azコマンドのパスを検索
az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
        print(f"🔍 Azure CLI発見: {result.stdout.strip()}")
except:
    pass

# 発見されたパスまたは可能なパスをPATHに追加
paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    # whichで見つからない場合は可能なパスを追加
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path
    print(f"✅ PATHに追加されたパス: {', '.join(paths_to_add)}")
else:
    print("✅ PATH가 이미 올바르게 設定되어 있습니다.")

print(f"\n💡 現在のPATH（最初の150文字）: {os.environ['PATH'][:150]}...")

In [ ]:
# Azure認証（Python SDK使用）
from azure.identity import InteractiveBrowserCredential, DeviceCodeCredential
from azure.mgmt.resource import SubscriptionClient
import os

# 테넌트 ID 設定 (필요시 変更)
TENANT_ID = os.getenv("AZURE_TENANT_ID", "16b3c013-d300-468d-ac64-7eda0820b6d3")

# GitHub Codespacesまたはリモート環境の検出
IS_CODESPACES = os.getenv("CODESPACES") == "true" or os.getenv("CODESPACE_NAME") is not None
IS_REMOTE = os.getenv("REMOTE_CONTAINERS") == "true" or IS_CODESPACES

print("🔐 Azure 認証 開始...")
print(f"Tenant ID: {TENANT_ID}")

if IS_REMOTE:
    print("\n💡 リモート環境（Codespaces/Remote Container）が検出されました。")
    print("デバイスコード認証を使用します。\n")
else:
    print("ブラウザが開いたらAzureアカウントでログインしてください。\n")

try:
    # 環境에 따라 다른 認証 방식 使用
    if IS_REMOTE:
        # Codespaces/Remote: DeviceCodeCredential 使用
        credential = DeviceCodeCredential(tenant_id=TENANT_ID)
        print("📱 以下の手順に従って認証してください:")
        print("   1. 以下のURLをローカルブラウザで開きます")
        print("   2. 表示されるコードを入力します")
        print("   3. Azureアカウントでログインします\n")
    else:
        # 로컬: InteractiveBrowserCredential 使用
        credential = InteractiveBrowserCredential(tenant_id=TENANT_ID)
    
    # サブスクリプション リスト 取得
    subscription_client = SubscriptionClient(credential)
    subscriptions = list(subscription_client.subscriptions.list())
    
    print("✅ Azure 認証 完了!")
    print("\n" + "=" * 80)
    print("📋 利用可能なサブスクリプション一覧:")
    print("=" * 80)
    
    for i, sub in enumerate(subscriptions, 1):
        print(f"\n{i}. {sub.display_name}")
        print(f"   Subscription ID: {sub.subscription_id}")
        print(f"   状態: {sub.state}")
    
    print("\n" + "=" * 80)
    print(f"✅ 合計 {len(subscriptions)}個のサブスクリプションが見つかりました。")
    
    # デフォルト サブスクリプション 情報 保存 (첫 번째 サブスクリプション)
    if subscriptions:
        default_sub = subscriptions[0]
        print(f"\n💡 デフォルトサブスクリプション: {default_sub.display_name}")
        print(f"   Subscription ID: {default_sub.subscription_id}")
        
        # 環境 変数로 保存
        os.environ["AZURE_SUBSCRIPTION_ID"] = default_sub.subscription_id
        print("\n✅ デフォルト サブスクリプション ID가 環境 変数에 保存되었습니다.")
    
except Exception as e:
    print(f"\n⚠️ 認証 失敗: {e}")
    print("\n💡 解決方法:")
    print("   1. 테넌트 ID가 올바른지 確認")
    if IS_REMOTE:
        print("   2. 디바이스 コード 認証 URL을 로컬 브라우저에서 열었는지 確認")
        print("   3. 表示된 コード를 정확히 入力했는지 確認")
    else:
        print("   2. 브라우저에서 ログイン 完了 確認")
    print("   4. アカウント에 해당 테넌트 アクセス 権限이 있는지 確認")

## Resource Group 作成

Resource GroupはAzureリソースを論理的にグループ化するコンテナです。

In [ ]:
# Resource Group 作成
!az group create \
    --name foundry-code \
    --location swedencentral

# 作成 確認
!az group show \
    --name foundry-code \
    --output table

## Foundry リソース 作成

고유한 名前으로 Foundry(AIServices) リソース를 作成합니다.

In [ ]:
# 環境 変数 設定
import os
import random
import string

# 고유한 名前 作成
random_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
FOUNDRY_NAME = f"foundry-{random_suffix}"
PROJECT_NAME = "default-project"
RESOURCE_GROUP = "foundry-code"
LOCATION = "swedencentral"

print(f"✅ 環境 変数 設定")
print(f"   Foundry: {FOUNDRY_NAME}")
print(f"   Project: {PROJECT_NAME}")
print(f"   Resource Group: {RESOURCE_GROUP}")
print(f"   Location: {LOCATION}")

In [ ]:
# Step 1: Foundry Resource 作成 (AIServices)
import subprocess
import json

subscription_id = os.environ.get('AZURE_SUBSCRIPTION_ID', '')

# 🔧 修正: API バージョン을 2025-04-01-preview로 変更 (allowProjectManagement 지원)
foundry_url = f"https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_NAME}?api-version=2025-04-01-preview"

foundry_body = json.dumps({
    "location": LOCATION,
    "kind": "AIServices",
    "sku": {"name": "S0"},
    "identity": {"type": "SystemAssigned"},
    "properties": {
        "customSubDomainName": FOUNDRY_NAME,
        "publicNetworkAccess": "Enabled",
        "allowProjectManagement": True
    }
})

print(f"📌 Step 1: Foundry Resource 作成 중: {FOUNDRY_NAME}")
print("   （AIServicesタイプ、API v2025-04-01-preview）")
print("   ✅ allowProjectManagement: True")
print("   💡 disableLocalAuth: 設定하지 않음 (Azure デフォルト値 適用)")

result = subprocess.run(
    ['az', 'rest', '--method', 'PUT', '--url', foundry_url, '--body', foundry_body],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ Foundry Resource 作成 完了")
    foundry_info = json.loads(result.stdout)
    foundry_id = foundry_info.get('id', '')
    os.environ["FOUNDRY_ID"] = foundry_id
    
    # allowProjectManagement 確認
    properties = foundry_info.get('properties', {})
    allow_project = properties.get('allowProjectManagement')
    disable_local_auth = properties.get('disableLocalAuth')
    
    print(f"   Foundry ID: {foundry_id[:70]}...")
    print(f"   allowProjectManagement: {allow_project}")
    print(f"   disableLocalAuth: {disable_local_auth}")
    
    if allow_project and not disable_local_auth:
        print("   ✅ Project 作成 및 API Key 認証 준비 完了")
    else:
        print(f"   ⚠️ 設定 確認 필요")
else:
    print(f"⚠️ Foundry 作成 失敗: {result.stderr}")

In [ ]:
# Step 2: Foundry Project 作成 (Subresource)
import subprocess
import json
import time

print(f"📌 Step 2: Foundry Project 作成 중: {PROJECT_NAME}")

foundry_id = os.environ.get('FOUNDRY_ID', '')
if foundry_id:
    # Foundry Resource 作成 完了 待機
    time.sleep(5)
    
    # Project를 Foundry의 subresource로 作成
    project_url = f"https://management.azure.com{foundry_id}/projects/{PROJECT_NAME}?api-version=2025-04-01-preview"
    project_body = json.dumps({
        "location": LOCATION,
        "identity": {"type": "SystemAssigned"},
        "properties": {
            "friendlyName": PROJECT_NAME,
            "description": f"Foundry Project: {PROJECT_NAME}"
            # disableLocalAuth 設定 削除 - Azure デフォルト値 確認
        }
    })
    
    print("   💡 API Key 認証: 設定하지 않음 (Azure デフォルト値 適用)")
    
    result = subprocess.run(
        ['az', 'rest', '--method', 'PUT', '--url', project_url, '--body', project_body],
        capture_output=True, text=True
    )
    
    if result.returncode == 0:
        print("✅ Foundry Project 作成 完了")
        project_info = json.loads(result.stdout)
        project_id = project_info.get('id', '')
        os.environ["PROJECT_ID"] = project_id
        print(f"   Project ID: {project_id[:70]}...")
        
        # disableLocalAuth 確認
        properties = project_info.get('properties', {})
        disable_local_auth = properties.get('disableLocalAuth', True)
        if not disable_local_auth:
            print("   ✅ API Key認証が有効化されました")
        else:
            print("   ⚠️ API Key認証がまだ無効状態です")
    else:
        print(f"⚠️ Project 作成 失敗: {result.stderr}")
else:
    print("⚠️ FOUNDRY_IDがありません。 前へ 셀을 먼저 実行하세요.")

In [ ]:
# Foundry Resource 및 Project 確認
import subprocess

print("📋 作成된 リソース 確認:\n")

result = subprocess.run(
    ['az', 'cognitiveservices', 'account', 'show',
     '--name', FOUNDRY_NAME,
     '--resource-group', RESOURCE_GROUP,
     '--query', '{Name:name, Kind:kind, Location:location, Endpoint:properties.endpoint}',
     '--output', 'table'],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("Foundry Resource:")
    print(result.stdout)
    print(f"\n✅ Foundry Resource와 Project가 作成되었습니다!")
    print(f"💡 Portal: https://ai.azure.com")
else:
    print(f"⚠️ 確認 失敗: {result.stderr}")

## API KeyとEndpointの取得

FoundryリソースのAPI KeyとEndpointを取得します。

In [ ]:
# API Key 作成 (Foundry Resource용)
import subprocess
import json

print(f"📌 Foundry API Key 作成")
print("💡 Foundry Projectは親AIServicesリソースのKeyを使用します。\n")

foundry_id = os.environ.get('FOUNDRY_ID', '')
if foundry_id:
    # 방법 1: Azure CLI 명령어로 Key インポート
    print("🔑 方法1: Azure CLIでKeyを取得")
    result = subprocess.run(
        ['az', 'cognitiveservices', 'account', 'keys', 'list',
         '--name', FOUNDRY_NAME,
         '--resource-group', RESOURCE_GROUP],
        capture_output=True, text=True
    )
    
    if result.returncode == 0:
        keys = json.loads(result.stdout)
        primary_key = keys.get('key1', '')
        
        if primary_key:
            print("✅ API Key 取得 完了")
            print(f"   Key1: {primary_key[:20]}...")
            os.environ["FOUNDRY_API_KEY"] = primary_key
            
            # Endpointも取得
            endpoint_result = subprocess.run(
                ['az', 'cognitiveservices', 'account', 'show',
                 '--name', FOUNDRY_NAME,
                 '--resource-group', RESOURCE_GROUP,
                 '--query', 'properties.endpoint',
                 '--output', 'tsv'],
                capture_output=True, text=True
            )
            
            if endpoint_result.returncode == 0:
                base_endpoint = endpoint_result.stdout.strip()
                # Project エンドポイント 構成
                # cognitiveservices.azure.comをservices.ai.azure.comに変更
                project_endpoint = base_endpoint.replace(
                    ".cognitiveservices.azure.com/",
                    ".services.ai.azure.com/"
                ).rstrip('/') + f"/api/projects/{PROJECT_NAME}"
                
                os.environ["FOUNDRY_ENDPOINT"] = project_endpoint
                print(f"   Base Endpoint: {base_endpoint}")
                print(f"   Project Endpoint: {project_endpoint}")
        else:
            print("⚠️ Keyが見つかりません。")
    else:
        print(f"⚠️ Key 取得 失敗: {result.stderr}")
        print("\n🔑 方法2: REST APIでKey取得を試行")
        
        # 방법 2: REST API로 시도
        key_url = f"https://management.azure.com{foundry_id}/listKeys?api-version=2025-04-01-preview"
        
        result2 = subprocess.run(
            ['az', 'rest', '--method', 'POST', '--url', key_url],
            capture_output=True, text=True
        )
        
        if result2.returncode == 0:
            keys = json.loads(result2.stdout)
            primary_key = keys.get('key1', '') or keys.get('primaryKey', '')
            
            if primary_key:
                print("✅ API Key 取得 完了 (REST API)")
                print(f"   Key: {primary_key[:20]}...")
                os.environ["FOUNDRY_API_KEY"] = primary_key
            else:
                print("⚠️ Keyが見つかりません。")
        else:
            print(f"⚠️ REST API도 失敗: {result2.stderr}")
else:
    print("⚠️ FOUNDRY_IDがありません。")

print(f"\n💡 Portal에서 確認: https://ai.azure.com")
print(f"💡 またはAzure Portalで '{FOUNDRY_NAME}' リソース의 Keys and Endpoint 메뉴 確認")


### Azure Portal에서 確認 (選択사항)

作成된 リソース를 시각적으로 確認하려면:

1. [Azure Portal](https://portal.azure.com)にアクセス
2. Resource Group `foundry`を検索
3. Foundry リソース 클릭하여 詳細 情報 確認

또는 [Microsoft Foundry Portal](https://ai.azure.com)에서 プロジェクト를 確認할 수 있습니다.

## 環境 変数 保存

次へ ノート북에서 使用할 주요 変数들을 ファイル로 保存합니다.

In [ ]:
# 環境 変数를 JSON ファイル로 保存
import json

config = {
    "FOUNDRY_NAME": FOUNDRY_NAME,
    "PROJECT_NAME": PROJECT_NAME,
    "RESOURCE_GROUP": RESOURCE_GROUP,
    "LOCATION": LOCATION,
    "AZURE_SUBSCRIPTION_ID": os.environ.get("AZURE_SUBSCRIPTION_ID", ""),
    "TENANT_ID": TENANT_ID,
    "FOUNDRY_ID": os.environ.get("FOUNDRY_ID", ""),
    "PROJECT_ID": os.environ.get("PROJECT_ID", ""),
    "FOUNDRY_API_KEY": os.environ.get("FOUNDRY_API_KEY", ""),
    "FOUNDRY_ENDPOINT": os.environ.get("FOUNDRY_ENDPOINT", "")
}

config_file = ".foundry_config.json"
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ 設定 ファイル 保存: {config_file}")
print(f"\n📋 保存된 情報:")
print(f"   Foundry: {FOUNDRY_NAME}")
print(f"   Project: {PROJECT_NAME}")
print(f"   Location: {LOCATION}")
print(f"   Endpoint: {os.environ.get('FOUNDRY_ENDPOINT', 'N/A')}")
print(f"   API Key: {'設定됨' if os.environ.get('FOUNDRY_API_KEY') else '미設定'}")
print(f"   タイプ: Foundry Project（AIServices + Project subresource）")
print(f"\n💡 次へ ノート북에서 이 設定을 자동으로 ロード합니다.")


## 📚 追加リソース

- [Microsoft Foundryドキュメント](https://learn.microsoft.com/en-us/azure/ai-foundry/what-is-azure-ai-foundry?view=foundry)
- [Azure Resource Manager概要](https://learn.microsoft.com/azure/azure-resource-manager/management/overview)
- [Azureリージョンと可用性ゾーン](https://learn.microsoft.com/azure/reliability/availability-zones-overview)

## 次のステップ

環境設定이 完了되었습니다! 이제 次へ モジュール로 進行하세요:

➡️ **[02. モデル 및 デプロイ](./02-models.ipynb)**: 다양한 AI モデル을 탐색하고 デプロイ하는 방법을 학습합니다.